#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

In [2]:
DATA_DIR = r"C:\Users\yugil\Downloads\Cassava_Leaf_Datasets\Cassava_Leaf_Datasets\Classification"

WEIGHTS_DIR = "../weights"

BATCH_SIZE = 32
FREEZE_EPOCHS = 8
FINETUNE_EPOCHS = 20

LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [3]:
weights = MobileNet_V3_Large_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])


In [4]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


Classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [5]:
model = mobilenet_v3_large(weights=weights)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to C:\Users\yugil/.cache\torch\hub\checkpoints\mobilenet_v3_large-5c1a4163.pth
100%|██████████| 21.1M/21.1M [00:00<00:00, 25.8MB/s]


In [6]:
for param in model.parameters():
    param.requires_grad = False


In [7]:
in_features = model.classifier[-1].in_features

model.classifier[-1] = nn.Linear(in_features, num_classes)

model = model.to(DEVICE)

In [8]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=LR
)


In [9]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss, train_acc


In [10]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss, val_acc

#### Training 1

In [11]:

print("\n-----------Starting Phase 1 Training-----------\n")

for epoch in range(FREEZE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FREEZE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FREEZE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")




-----------Starting Phase 1 Training-----------



Validating: 100%|██████████| 8/8 [00:30<00:00,  3.78s/it]



Epoch [1/8]
Train Loss: 39.3633 | Train Acc: 0.4458
Val Loss: 10.6720 | Val Acc: 0.2958
Precision: 0.3128 | Recall: 0.2958 | F1: 0.1946


Validating: 100%|██████████| 8/8 [00:30<00:00,  3.83s/it]



Epoch [2/8]
Train Loss: 36.1374 | Train Acc: 0.6583
Val Loss: 10.1354 | Val Acc: 0.3917
Precision: 0.4539 | Recall: 0.3917 | F1: 0.3364


...


Epoch [6/8]
Train Loss: 26.8175 | Train Acc: 0.8260
Val Loss: 7.3582 | Val Acc: 0.7750
Precision: 0.7861 | Recall: 0.7750 | F1: 0.7741


Validating: 100%|██████████| 8/8 [00:29<00:00,  3.74s/it]



Epoch [7/8]
Train Loss: 25.1834 | Train Acc: 0.8406
Val Loss: 6.8861 | Val Acc: 0.8000
Precision: 0.8094 | Recall: 0.8000 | F1: 0.7995


Validating: 100%|██████████| 8/8 [00:29<00:00,  3.70s/it]


Epoch [8/8]
Train Loss: 23.5648 | Train Acc: 0.8625
Val Loss: 6.5130 | Val Acc: 0.7958
Precision: 0.8005 | Recall: 0.7958 | F1: 0.7948


#### Training 2 (Fine-Tuning)

In [12]:
for param in model.features[-1].parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)


In [13]:
best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(FINETUNE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FINETUNE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FINETUNE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names
}, os.path.join(WEIGHTS_DIR, "MobileNet-v3.pth"))


-----------Starting Fine-tuning Stage-----------



Validating: 100%|██████████| 8/8 [00:29<00:00,  3.66s/it]



Epoch [1/20]
Train Loss: 22.9213 | Train Acc: 0.8500
Val Loss: 6.4273 | Val Acc: 0.8042
Precision: 0.8088 | Recall: 0.8042 | F1: 0.8033


Validating: 100%|██████████| 8/8 [00:29<00:00,  3.66s/it]



Epoch [2/20]
Train Loss: 22.7177 | Train Acc: 0.8531
Val Loss: 6.3447 | Val Acc: 0.8125
Precision: 0.8162 | Recall: 0.8125 | F1: 0.8119


...


Epoch [18/20]
Train Loss: 18.5054 | Train Acc: 0.8604
Val Loss: 5.3367 | Val Acc: 0.8375
Precision: 0.8386 | Recall: 0.8375 | F1: 0.8372


Validating: 100%|██████████| 8/8 [00:29<00:00,  3.63s/it]



Epoch [19/20]
Train Loss: 17.9970 | Train Acc: 0.8708
Val Loss: 5.2803 | Val Acc: 0.8375
Precision: 0.8386 | Recall: 0.8375 | F1: 0.8372


Validating: 100%|██████████| 8/8 [00:29<00:00,  3.63s/it]


Epoch [20/20]
Train Loss: 17.9782 | Train Acc: 0.8729
Val Loss: 5.2333 | Val Acc: 0.8375
Precision: 0.8386 | Recall: 0.8375 | F1: 0.8372


#### Prediction Sample

In [14]:
checkpoint = torch.load(r"C:\Users\e6va6je238\Desktop\CassavAI\model\weights\MobileNet-v3.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]


C:\Users\yugil\AppData\Local\Temp\ipykernel_22708\3148693027.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(r"C:\Users\e6va6je238\Desktop\Cassav

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\e6va6je238\\Desktop\\CassavAI\\model\\weights\\MobileNet-v3.pth'

In [ ]:
prediction = predict(r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val\Healthy\Healthy_val_13.jpg")
prediction1 = predict(r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val\leaf blight\leaf blight_val_13.jpg")
prediction2 = predict(r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val\leaf spot\leaf spot_val_13.jpg")
prediction3 = predict(r"C:\Users\e6va6je238\Desktop\Cassava_Leaf_Datasets\Classification\val\Spider Mites\Spider Mites_val_13.jpg")
print("Healthy Predicted Class:", prediction)
print("Leaf blight Predicted Class:", prediction1)
print("Leaf spot Predicted Class:", prediction2)
print("Spider Mites Predicted Class:", prediction3)


Healthy Predicted Class: Spider Mites
Leaf blight Predicted Class: leaf blight
Leaf spot Predicted Class: leaf spot
Spider Mites Predicted Class: Spider Mites
